In [1]:
%run cochain_complex.ipynb
import copy
import random
import decimal
from sympy import *
from multiset import *
from itertools import combinations
import time 

# Initial Structure Function

We'll start with a distribution with structure function $K$, then prolong and normalize. \
The bundle $P_0$ will have vertical coordinates $h$ and $e$;\
The bundle $P_1$ will have vertical coordinate $y$.

In [2]:
NF_dict={}
m=3
n=m+3
K=IndexedBase('K')
h,e,y=symbols('h,e,y')
T=T_symb(2*m+1)
C=T.cochain_complex

In [3]:
# Start with the structure function on the base manifold; the H,E,Y components don't have meaning here
K0=C.cochain({})
C.init_basis(2)
for wght in C.basis[2]:
    if wght>=0:
        for c in C.basis[2][wght]:
            t=C.ijk_triple(c)
            if t[0]>2 and t[1]>2 and t[2]>2:
                A=[T.basis_strs[ti] for ti in t]
                K0=K0+C.cochain({(A[0],A[1],A[2]):K[t[0],t[1],t[2]]})
                
# bracket relations (degree zero part of structure function)
br_rels={}
br_rels.update({K[3,a,a+1]:1 for a in range(4,2*m+3)}) # [X,ei]
br_rels.update({K[3,2*m+3,2*m+4]:0}) # Last [X,ei]
br_rels.update({K[a,2*m+7-a,2*m+4]:(-1)**(1+a) for a in range(4,m+4)}) # [ei, e_(2m+1-i)]
br_rels.update({K[a,b,a+b-3]:0 for a in range(4,2*m+4) for b in range(4,2*m+7-a)}) #[ei,ej], i+j<2m+1

K0=K0.subs(br_rels)

## Exceptions

In [4]:
class SubsFailureException(Exception):
     def __init__(self, B, B_subs, message="No substitution for tuple"):
        self.message = message + str((B,B_subs))
        super().__init__(self.message)

In [5]:
class HeuristicFailureException(Exception):
    def __init__(self,Rel,message="Heuristic Algorithm in Subs_From_Rel failed"):
        super().__init__(self.message)

## Prenormalization

From the involutivity conditions $[V_i,V_i]\subseteq V_i$ and $[\mathcal{J}^{(i)},V_i]\subseteq \mathcal{J}^{(i)}$, we can impose the following conditions on the structure function $K_{ij}^k$:

$$ [\varepsilon_i,\varepsilon_j]=\sum_{k=1}^j K_{ij}^k\varepsilon_k,\quad \text{for}\ i<j\leq m$$
$$ [\varepsilon_i,\varepsilon_j]=K_{ij}^XX+\sum_{k=1}^j K_{ij}^k\varepsilon_k,\quad \text{for}\ j>m\ \text{and}\ i+j<2m+1$$
All this is captured in the below "prenormalization", but the indices are shifted to reflect the index of each element in the basis $(Y,H,E,X,\varepsilon_1,\ldots \varepsilon_6,\eta)$.

In [6]:
# We can impose the following conditions on the structure function K
prenorm_dict=copy.copy(br_rels)

# involutivity relations
# First relation
prenorm_dict.update({K[a,b,c]:0 for b in range(4,m+4)
                     for a in range(4,b) for c in range(b+1,a+b+1)})
prenorm_dict.update({K[a,b,3]:0 for b in range(4,m+4) for a in range(4,b)})

# Second relation
prenorm_dict.update({K[a,b,c]:0 for b in range(m+4,2*m+4)
                    for a in range(4,2*m+7-b) for c in range(b+1,a+b-3)})
K0=K0.subs(prenorm_dict)

## Jacobi Relations

In [7]:
def Rat_Solve(Rel,X):
    '''args: Rel, a rational Relession in symbols and Indexed objects
       X: a symbol or Indexed object
       returns: By degree of Rel in X:
                0) True if zero, False otherwise
                1) A list with a single root of Rel
                2) A list of both roots of Rel
                3) None'''
    # At the moment, this can't deal with degree higher than 2
    N=reduce_numer(Rel)
    try: d = degree(N,X)
    except PolynomialError:
        print('PolynomialError: degree of', (N,X),'not determined')

    if N==0: return True
    if d==0: return False
    if d==1:
        num=-N.subs(X,0)
        den=N.subs(X,1)+num
        result=[simplify(num/den)]
    if d==2:
        C=simplify(N.subs(X,0))
        B=simplify((N-C)/X).subs(X,0)
        A=simplify((N-C-B*X)/X**2)

        S1=(-B+sqrt(B**2-4*A*C))/(2*A)
        S2=(-B-sqrt(B**2-4*A*C))/(2*A)
        result = [S1,S2]
    if d>2:
        print('Rat_Solve error: degree of', (N,X),'is greater than 2')
        return None
        
    # check the results
    # Maybe this is too costly to keep
    for r in result:
        if simplify(Rel.subs(X,r))!=0:
            result.remove(r)
    if len(result)==0:
        print('No solutions from Rat_Solve:', (Rel,X))
    return result 

In [8]:
def is_poly(expr):
    '''arg: expr, a simplified algebraic expression in some Indexed objects
    returns True if expr is polynomial, False otherwise'''
    if type(expr)==int: return True
    SD={A:symbols(str(A)) for A in Indexed_factors(expr)}
    return expr.subs(SD).is_polynomial(*SD.values())

In [9]:
def Rationalize_Rel(expr,IndObj=True):
    '''arg: expr, an algebraic expression in some symbols; IndObj, False if expr involves no Ind_Obj
       returns: a polynomial expression whose vanishing implies the vanishing of expr'''
    # In order to avoid substituting at each is_poly() call,
    # substitute all Indexed objs out from the outset
    if IndObj==True:
        SD1={A:symbols(str(A)) for A in Indexed_factors(expr)}
        SD2={SD1[A]:A for A in SD1}
        NE=expr.subs(SD1)
    else: NE=expr
        
     
    if is_poly(NE):
        result=NE
    elif type(NE)==Pow:
        result=Rationalize_Rel(NE.as_base_exp()[0],IndObj=False)
    elif type(NE)==Mul:
        result=prod([Rationalize_Rel(A) for A in NE.as_coeff_mul()[1]])
    elif type(NE)==Add:
        # For now, assume NE is a sum of one polynomial term and one square root
        poly_term = NE.as_coeff_add()[0]
        rad_term = 0
        for A in NE.as_coeff_add()[1]:
            if is_poly(A):
                poly_term+=A
            else:
                rad_term+=A
        if len(rad_term.as_coeff_add()[1])>1:
            print('Rationalise_Rel Failure: Too many radical terms in NE =', NE)
            print('poly_term =',poly_term)
            print('rad_term =',rad_term)
        result=Rationalize_Rel(rad_term**2-poly_term**2)
    
    if IndObj==True: 
        result=result.subs(SD2) # Substitute back in for the symbols
    return result

In [10]:
def Subs_From_Rel(Rel,subs_dict,init=False):
    '''Rel: Some relation in the K
       subs_dict: a dictionary of substitutions
       returns: True if the Relation is trivial,
                False if the Relation is invalid,
                otherwise (A, A_subs) such that Rel==0 implies A==A_subs
                where A is a K[] not in subs_dict.'''
    R=reduce_numer(Rel)

    # Check if the relation is just a number
    if R==0:return True
    if type(R) in [int,float,Rational]: return False

    # Choose a new elem
    NEL=Indexed_factors(R)
    NEL.difference_update(set(subs_dict.keys())) # New Element List

    ctr=0
    # If no new elts are involved, substitute and look again
    while NEL==set() and ctr<20:
        if ctr>0: print('ctr =',ctr)
        R=simplify(R.subs(subs_dict))
        
        # Check again if the relation is just a number
        sR=simplify(R)
        if sR==0:return True
        if type(sR) in [int,float,Rational]: return False
        
        NEL=Indexed_factors(R)
        NEL.difference_update(set(subs_dict.keys()))
        
    if ctr==20: raise HeuristicFailureException(R) # The above loop may not terminate (but it should)
    
    # Ideally, choose those elements in few relations
    NEL=sorted(list(NEL),key=(lambda A: J_count[A]))
    
    # We'd like to avoid radicals if possible
    d1Elt=None
    for A in NEL:
        try:
            if degree(R,A)==1:
                d1Elt=A
                break
        except PolynomialError: pass
    if d1Elt!=None: NE=d1Elt
    else: NE=NEL[0]
    NR_sols=Rat_Solve(R,NE)

    if type(NR_sols)==list:
        if len(NR_sols)==0:
            raise SubsFailureException(NE,R,message='Rat_Solve cannot solve'+str(Rel)+'for'+str(NE))
        if len(NR_sols)>1:
            print('Multiple solutions for new Index', NE)
        return (NE,NR_sols[0])
    raise SubsFailureException(None,R,message='Subs_From_Rel fails')

In [11]:
def Make_val_subs(A,A_subs,subs_dict):
    '''Substitutes A_subs in for A in all values of subs_dict, updating the keys of subs_dict
       returns: A set of all B in subs_dict so that subs_dict[B] involves B'''
    result=set()
    for B in list(subs_dict.keys()):
        if A in Indexed_factors(subs_dict[B]):
            subs_dict[B]=subs_dict[B].subs(A,subs_dict[A])
            if B in Indexed_factors(subs_dict[B]): result.add(B)
    return result

In [12]:
def Make_all_JR_subs():
    time0=time.time()
    ctr=1
    KS=set(JR_subs.keys())
    # Set of keys to iterate through, subbing out as we go.
    #len(KS) decreases by at least one in every loop
    while len(KS)!=0:
        time1=time.time()
        k=next(iter(KS))
        R=Make_val_subs(k,JR_subs[k],JR_subs)
        
        # remove keys for reprocessing
        KS.difference_update(R)
        for B in R:
            Curr_Rel=B-JR_subs[B]
            del JR_subs[B]
            S=Subs_From_Rel(Curr_Rel,JR_subs)
            if type(S)==bool:
                if not S: raise SubsFailureException(B,Curr_Rel)
            else:
                JR_subs[S[0]]=S[1]
                KS.add(S[0])
                
        # k has been eliminated from JR_subs, except as a key
        KS.discard(k)
        time2=time.time()
#         print('key',ctr,' =',k,'finished in time',round(time2-time1,2),'sec')
        ctr+=1
    print('All JR_subs made. Total time:',round(time.time()-time0),'sec')

In [13]:
# Jacobi relations:
JR_dict={}

for i in range(3,len(T.basis)):
    Ai=T.basis[i]
    for j in range(i+1,len(T.basis)):
        Aj=T.basis[j]
        for k in range(j+1, len(T.basis)):
            Ak=T.basis[k]
            R1=SF_ad(Ai,SF_ad(Aj,Ak,K0),K0)
            R2=SF_ad(SF_ad(Ai,Aj,K0),Ak,K0)+SF_ad(Aj,SF_ad(Ai,Ak,K0),K0)
            v=(R1-R2).vec_rep
            for l in range(len(v)):
                if v[l]!=0:
                    JR_dict[(i,j,k,l)] = v[l]

JR_list=list(JR_dict.values())

In [14]:
# Count the number of times is K[] shows up in the JR

J_count={}
for R in JR_list:
    RF=Indexed_factors(R)
    for A in RF:
        if A in J_count: J_count[A]+=1
        else: J_count[A]=1

In [15]:
# Inital substitution dict JR_subs
# Approx. 1 min.

# Heuristic algorithm: Choose the Indexed objects to isolate according to # of options
# If an exception is thrown, the heuristic algorithm failed;
# that is, one of the relations brings no new Indexed object to the table
time0=time.time()
JR_subs={}
JR_list.sort(key=lambda a:len(Indexed_factors(a)))
for J in JR_list:
    
    ## Choose those that show up in few JRs as indep vars 
    S=Subs_From_Rel(J,JR_subs,init=True)
    if type(S)==bool:
        if not S: print('Inconsistent relation:', J)
    else: JR_subs[S[0]]=S[1]
JR_subs_OG=copy.copy(JR_subs) # For sanity check 2
print('Inital Jacobi substitutions set. Total time:', hrs_min_sec(time.time()-time0),'sec')

Inital Jacobi substitutions set. Total time: 57.0 sec sec


In [16]:
# Approx. 15 sec
Make_all_JR_subs()

All JR_subs made. Total time: 12 sec


In [17]:
# Approx. 10 sec
time0=time.time()
K0.subs(JR_subs)
print('Subs made. Total time',hrs_min_sec(time.time()-time0))

Subs made. Total time 8.0 sec


# Computing Geometric Prolongations

### Initialization

In [18]:
# Approx. 10 sec

time0=time.time()
F0=eye(len(T.basis))
F0[0,0]=0

# ift invariantly; no need to normalize degree zero
F0=F0*T.Ad_mat(h*T.basis[1]+e*T.basis[2])

# Lift invariantly again
L=zeros(len(T.basis))
L[0,0]=1
F1=(F0+L)*T.Ad_mat(y*T.basis[0])
K1=str_func(T,F1,K0)
print('Step 0 total time', hrs_min_sec(time.time()-time0))

Step 0 total time 10.0 sec


### Approach 0: All degrees, not simultaneous, only structure functions

In [19]:
fi=[None]*8
dfi=[None]*8
KNi_degi=[None]+[None]*7
KNi_degip1=[K1]+[None]*7
FNi=[F1]+[None]*7

In [27]:
fi=[None]*8
dfi=[None]*8
KNi=[K1]+[None]*7 # Normal to degree equal to its index


def Normalize_SF(d):
    '''Normalizes the frame in degree d, assuming degree (d-1) is already normalized.
       Modifies fi, KNi_degi, KNi_degip1, and FNi'''
    time0=time.time()
    
    # Normalize
    dfi[d],df_coords=C.subspace_proj(KNi[d-1].wght_proj(d),'im')
    fi[d]=C.preim_elt(df_coords)
    simplify_cochain(fi[d])
    
    time1=time.time()
    print('f',d,'computed in time',hrs_min_sec(time1-time0))
    
    # Compute new structure function    
    KNi[d]=KNi[d-1].GL_action(eye(len(T.basis))-fi[d].find_mat_rep())
    simplify_cochain(KNi[d])
    
    print('Degree',d,'normalized in time',hrs_min_sec(time.time()-time1))

In [ ]:
for i in range(1,9):
    time0=time.time()
    Normalize_SF(i)
    print('Degree',i,'frame normalized in time', hrs_min_sec(time.time()-time0))

f 1 computed in time 20.0 sec


### Approach 1: Normalizing by degree

### Approach 1a: Only degree one

In [ ]:
# # Degree 1 Normalization only
# # Before JR_subs: 36 sec

# time0=time.time()

# # Normalize
# df1,df1_coords=C.subspace_proj(C.mg_proj(K1.wght_proj(1)),'im')
# f1=C.preim_elt(df1_coords)
# simplify_cochain(f1)
# print('Step 1 total time', hrs_min_sec(time.time()-time0))

In [ ]:
# # Before JR_subs: 4 min 14 sec
# time0=time.time()

# F1N=F1*(eye(len(T.basis))-f1.find_mat_rep())
# # Compute new structure function
# K1N_temp=str_func(T,F1N,K0)
# print('Step 2 total time', hrs_min_sec(time.time()-time0))

In [ ]:
# # Before JR_subs: 0 sec
# time0=time.time()
# K1N=C.mg_proj(K1N_temp.wght_proj(1))
# #simplify_cochain(K1N)
# print('Step 3 total time', hrs_min_sec(time.time()-time0))

### Approach 1b: All degrees, not simultaneous

In [ ]:
fi=[None]*8
dfi=[None]*8
KNi_degi=[None]+[None]*7
KNi_degip1=[K1]+[None]*7
FNi=[F1]+[None]*7

In [ ]:
def Normalize_Frame(d):
    '''Normalizes the frame in degree d, assuming degree (d-1) is already normalized.
       Modifies fi, KNi_degi, KNi_degip1, and FNi'''
    # Normalize
    dfi[d],df_coords=C.subspace_proj(KNi_degip1[d-1].wght_proj(d),'im')
    fi[d]=C.preim_elt(df_coords)
    simplify_cochain(fi[d])
    FNi[d]=FNi[d-1]*(eye(len(T.basis))-fi[d].find_mat_rep())

    # Compute new structure function
    KNi_degi[d]=C.mg_proj(str_func(T,FNi[d],K0).wght_proj(d))
    KNi_degip1[d]=C.mg_proj(str_func(T,FNi[d],K0).wght_proj(d+1))
    simplify_cochain(KNi_degi[d])
    simplify_cochain(KNi_degip1[d])

In [ ]:
for i in range(1,9):
    time0=time.time()
    Normalize_Frame(i)
    print('Degree',i,'frame normalized in time', hrs_min_sec(time.time()-time0))

#### Some Sanity Checks

In [ ]:
# # Sanity Check 0
# # The data from Normalize_Frame match the data from 
# # the cell "Degree 1 Normalization only"

# res_a='passed'
# res_b='passed'
# res_c='passed'

# if fi[1]!=f1: res_a='failed'
# if FNi[1]!=F1N: res_b='failed'
# if KNi_degi[1]!=K1N: res_c='failed'
# print('Sanity Check 0a '+res_a)
# print('Sanity Check 0b '+res_b)
# print('Sanity Check 0c '+res_c)

In [ ]:
# # Sanity Check 1
# # str.func. transformation law

# for i in range(1,2):
#     res='passed'
#     test1=KNi_degi[i]-KNi_degip1[i-1]+dfi[i]
#     simplify_cochain(test1)
#     if test1!=0: res='failed'
#     print('Sanity Check 1 deg',i,' ',res)

In [ ]:
# # Sanity Check 1
# # str.func. transformation law

# res='passed'
# test1=K1N-K1+df1
# simplify_cochain(test1)
# if test1!=0: res='failed'
# print('Sanity Check 1 deg',i,' ',res)

In [ ]:
# for i in range(1,2):
#     res='passed'
#     test1=KNi_degi[i]-KNi_degip1[i-1]+fi[i].coboundary()
#     print('check 1')
#     simplify_cochain(test1)
#     print('check 2')
#     if test1!=0: res='failed'
#     print('Sanity Check 1 deg',i,' ',res)

In [ ]:
# # Sanity Check 2
# # KNi has invariants in degrees 3,4,6 (for n=6)

# for i in range(1,2):
#     res='passed'
#     test2=C.harm_proj(KNi_degi[i])
#     if test2!=0: res='failed'
#     print('Sanity Check 2 deg',i,' ',res)

In [ ]:
# # Sanity Check 3
# # dfi is the image of fi
# for i in range(1,2):
#     res='passed'
#     test3=fi[i].coboundary()-dfi[i]
#     simplify_cochain(test3)
#     if test3!=0: res='failed'   
#     print('Sanity Check 3 deg ',i,' ',res)

In [ ]:
# # Sanity Check 4
# # Ki[i-1] projects to dfi
# # This is basically how we defined it, just without the wght proj
# for i in range(1,2):
#     res='passed'
#     K_Im=C.subspace_proj(KNi_degip1[i-1],'im')[0]
#     test4=dfi[i]-K_Im.wght_proj(i)
#     simplify_cochain(test4)
#     if test4!=0: res='failed'
#     print('Sanity Check 4 deg ',i,' ',res)

#### Comparison of Approaches 1a, 1b

In [ ]:
# # temp
# def Compute_Preim(d):
#     dfi[d],df_coords=C.subspace_proj(KNi_degip1[d-1].wght_proj(d),'im')
#     fi[d]=C.preim_elt(df_coords)
#     simplify_cochain(fi[d])

In [ ]:
# # temp
# def Compute_SF(d):
#     FNi[d]=FNi[d-1]*(eye(len(T.basis))-fi[d].find_mat_rep())
#     nsf=str_func(T,FNi[d],K0)
#     KNi_degi[d]=C.mg_proj(nsf.wght_proj(d))
#     KNi_degip1[d]=C.mg_proj(nsf.wght_proj(d+1))
#     simplify_cochain(KNi_degi[d])
#     simplify_cochain(KNi_degip1[d])

In [ ]:
# # temp: Compute Preim Deg 1 only
# time0=time.time()

# df1,df1_coords=C.subspace_proj(C.mg_proj(K1.wght_proj(1)),'im')
# f1=C.preim_elt(df1_coords)
# simplify_cochain(f1)

# print('Complete. Time', hrs_min_sec(time.time()-time0))

In [ ]:
# # temp: Compute new SF Deg 1 only
# # Approx. 5 min
# time0=time.time()

# F1N=F1*(eye(len(T.basis))-f1.find_mat_rep())
# K1N=C.mg_proj(str_func(T,F1N,K0).wght_proj(1))
# simplify_cochain(K1N)

# print('Complete. Time', hrs_min_sec(time.time()-time0))

In [ ]:
# # temp: Compute Preim
# time0=time.time()
# Compute_Preim(1)
# print('Complete. Time', hrs_min_sec(time.time()-time0))

In [ ]:
# # temp: Compute new SF
# # Approx 6 min
# time0=time.time()
# Compute_SF(1)
# print('Complete. Time', hrs_min_sec(time.time()-time0))

### Approach 2: Normalizing simultaneously

In [ ]:
# # Approx.

# time0=time.time()
# # Normalize
# df,df_coords=C.subspace_proj(C.mg_proj(K1),'im')
# print('Total time:',hrs_min_sec(time.time()-time0))

In [ ]:
# # Approx.
# time0=time.time()
# f=C.preim_elt(df_coords)
# print('Total time:',hrs_min_sec(time.time()-time0))

In [ ]:
# # Approx.
# time0=time.time()
# simplify_cochain(f)
# print('Total time:',hrs_min_sec(time.time()-time0))

In [ ]:
# # Split into weighted pieces
# # Approx.
# time0=time.time()

# fw=[f.wght_proj(w) for w in range(len(T.basis)-2)]
# fw_mats=[eye(len(T.basis))-A.find_mat_rep() for A in fw]

# # Normalized frame
# cob_mat=prod(fw_mats)
# print('Total time:',hrs_min_sec(time.time()-time0))

In [ ]:
# # Compute the final structure function
# time0=time.time()
# KF = K1.GL_action(cob_mat)
# print('Total time:',hrs_min_sec(time.time()-time0))

In [ ]:
# # Compute the harmonic part of KF
# time0=time.time()
# K_harm=KF.harm_proj()
# print('Total time:',hrs_min_sec(time.time()-time0))

In [ ]:
# # Sanity Check 5
# # The graded pieces of f sum to f

# res='passed'

# test5=sum(fw)-f
# simplify_cochain(test5)

# if test5!=0: res='failed'    
# print('Sanity Check 5 '+res)

In [ ]:
test=(K1-K1_Im).large_subs(JR_subs)

In [ ]:
simplify_cochain(test)
test.remove_zeros()
test==0

In [ ]:
test.wght_proj(1)

In [ ]:
for A in JR_subs:
    print('\n',A,'-->',JR_subs[A])

In [ ]:
K1P=C.mg_proj(K1.wght_proj(1))
simplify_cochain(K1P)
print(K1P)

In [ ]:
K1P=K1-C.mg_proj(K1)
simplify_cochain(K1P)
K1P

In [ ]:
K0.wght_proj(1)

In [ ]:
K0.wght_proj(1).subs(JR_subs)

In [ ]:
K0_coeff_list=[]
for A in K0_temp.coeff_dict:
    mul_list=K0_temp.coeff_dict[A].as_ordered_factors()
    for B in mul_list: 
        if type(B)==Indexed: K0_coeff_list.append(B)

In [ ]:
K1_p,K1_p_im_coords=C.subspace_proj(K1,'im')

In [ ]:
test0=K1_p-K1
simplify_cochain(test0)

In [ ]:
F0=eye(len(T.basis))
F0[0,0]=0
#Lift invariantly
F1=F0*T.Ad_mat(h*T.basis[1]+e*T.basis[2])
K1=str_func(T,F1,K0).wght_proj(1).subs(prenorm_dict) #This should be in the image, so that the normalized sf is 0
K1_p1,K1_p1_im_coords=C.subspace_proj(K1,'im')

In [ ]:
K_set.

In [ ]:
New_K_set=set()
K_set=set()
for A in (K1-K1_p1).coeff_dict.values():
    K_set=K_set.union(Indexed_factors(A))
    for B in K_set:
        if type(B)==type(K[0,0,0]): New_K_set.add(B)

In [ ]:
test0=(K1-K1_p1).subs(JR_subs)

In [ ]:
test0

In [ ]:
# This is the image in wght 1
for A in C.coboundary_im[2][1]:
    print(A,'\n')

In [ ]:
# simplify_cochain(K1)
# simplify_cochain(K1_p1)
print('\n\n\n')
for A in K1.coeff_dict:
    print(A,':\n')
    print('    ',K1.coeff_dict[A],'\n')
    #print('    ',K1_p1.coeff_dict[A],'\n\n\n')

In [ ]:
## Let's check using rank if K1 and K1_p1 is in the image
deg=2
wght=1
ims=C.coboundary_im_vecs[deg][wght]

K1_p1_coords=Matrix([K1_p1_im_coords[deg][wght]])*Matrix(ims)
wd_Mat=Matrix(C.coboundary_im_vecs[deg][wght]+[list(K1_p1_coords)])
print('rank:',wd_Mat.rank())
print('coboundary image dimension:',len(C.coboundary_im[deg][wght]))
if wd_Mat.rank()==len(C.coboundary_im[deg][wght]): print('K1_p1 is in the image\n\n')
else: print('K1_p1 is NOT in the image\n\n')
        
# What about K1?
K1_coords=Matrix([coordinatize_cochain_in_basis(K1,C.basis_strs[2][1])])
wd_Mat=Matrix(C.coboundary_im_vecs[deg][wght]+[list(K1_coords)])

# pprint(Matrix(C.coboundary_im_vecs[deg][wght]))
print('rank:',wd_Mat.rank())
print('coboundary image dimension:',len(C.coboundary_im[deg][wght]))
if wd_Mat.rank()==len(C.coboundary_im[deg][wght]): print('K1 is in the image')
else: print('K1 is NOT in the image')

# Since these are not equal, something must be wrong with K1...

In [ ]:
test0=K1_p1-K1
simplify_cochain(test0)

In [ ]:
# Normalize
f1=C.preim_elt(K1_p1)
f1_Mat=f1.find_mat_rep()
F1_N=F1*(eye(len(T.basis))+f1_Mat)
K1_N=str_func(T,F1_N,K0)

## Test of Jacobi Relations (Incomplete)

In [ ]:
# Sanity Check 1
# keys do not appear in values

time0=time.time()

for A in JR_subs:
    for B in JR_subs:
        if A in Indexed_factors(JR_subs[B]):
            print('Failure:',(A,B))

print('Sanity check complete. Total time:', hrs_min_sec(time.time()-time0),'sec')

In [ ]:
# Sanity Check 2
# JR_list becomes zero upon subbing JR_subs
time0=time.time()
failure_list=[]
for i in range(len(JR_list)):
    if JR_list[i].subs(JR_subs)!=0:
        failure_list.append(i)
if len(failure_list)>0:
    print('Failures:', failure_list)
print('Sanity check complete. Total time:', hrs_min_sec(time.time()-time0),'sec')

In [ ]:
# Sanity Check 3
# JR_subs and JR_sub_OG agree

time0=time.time()
for key in JR_subs_OG:
    if key in JR_subs:
        # Case 1: key is a dependent variable
        if JR_subs_OG[key].subs(JR_subs)!=JR_subs[key]:
            print('Failure 1:\nkey:',k,',\nJR_subs_OG[key].subs(JR_subs):',JR_subs_OG[key].subs(JR_subs),
                  '\nJR_subs[key]:',JR_subs[key])
    else:
        # Case 2: key is an independent variable
        if JR_subs_OG[key].subs(JR_subs)!=key:
            print('Failure 2:\nkey:',k,',\nJR_subs_OG[key].subs(JR_subs):',JR_subs_OG[key].subs(JR_subs))
print('Sanity check complete. Total time:', hrs_min_sec(time.time()-time0),'sec')    

In [ ]:
antisymm_subs={}

for i in range(3,len(T.basis)):
    for k in range(3,len(T.basis)):
        antisymm_subs[K[i,i,k]]=0
        for l in range(3,len(T.basis)):
            antisymm_subs[K[i,i,k,l]]=0

for i in range(3,len(T.basis)):
    Xi=str(T.basis[i])
    for j in range(i+1,len(T.basis)):
        Xj=str(T.basis[j])
        for k in range(3,len(T.basis)):
            Xk=str(T.basis[k])
            if C.tuple_wght((Xi,Xj,Xk))>=0:
                antisymm_subs[K[j,i,k]]=-K[i,j,k]
                for l in range(3,len(T.basis)):
                    antisymm_subs[K[j,i,k,l]]=-K[i,j,k,l]
            else:
                antisymm_subs[K[j,i,k]]=0
                antisymm_subs[K[i,j,k]]=0
                for l in range(3,len(T.basis)):
                    antisymm_subs[K[j,i,k,l]]=0
                    antisymm_subs[K[i,j,k,l]]=0
                    
prenorm_dict1=copy.copy(prenorm_dict)
pnd_keys=copy.copy(list(prenorm_dict.keys()))
for A in pnd_keys:
    Ai=A.indices
    for i in range(3,len(T.basis)):
        prenorm_dict1[K[Ai[0],Ai[1],Ai[2],i]]=0


In [ ]:
J_test_subs=copy.copy(antisymm_subs)
J_test_subs.update(br_rels)
J_test_subs.update(prenorm_dict1)

ctr=0
while True:
    print(ctr)
    J_test_subs1=copy.copy(J_test_subs)
    for A in J_test_subs:
        try: J_test_subs1[A]=J_test_subs1[A].subs(J_test_subs)
        except: pass
    if J_test_subs1==J_test_subs: break
    ctr+=1

In [ ]:
JR_check={}

for i in range(3,len(T.basis)):
    for j in range(i+1,len(T.basis)):
        for k in range(j+1, len(T.basis)):
            for l1 in range(3,len(T.basis)):
                rel=K[i,j,l1,k]+K[k,i,l1,j]+K[j,k,l1,i]
                for l2 in range(3,len(T.basis)):
                    rel+=-K[i,j,l2]*K[l2,k,l1]-K[j,k,l2]*K[l2,i,l1]-K[k,i,l2]*K[l2,j,l1]
                rel=rel.subs(J_test_subs)
                if rel!=0:
                    JR_check[(i,j,k,l1)]=rel

In [ ]:
len(JR_dict)

In [ ]:
len(JR_check)

In [ ]:
JR_K1=set(JR_dict.keys())
JR_K2=set(JR_check.keys())

missing=JR_K2
missing.difference_update(JR_K1)
missing=list(missing)

In [ ]:
missing

In [ ]:
antisymm_subs[K[5,7,10]]

In [ ]:
for i in [3]:
    for j in [5]:
        for k in [7]:
            for l1 in [9]:
                rel=-K[i,j,l1,k]-K[k,i,l1,j]-K[j,k,l1,i]
                for l2 in range(3,len(T.basis)):
                    rel+=K[i,j,l2]*K[l2,k,l1]+K[j,k,l2]*K[l2,i,l1]+K[k,i,l2]*K[l2,j,l1]
                    if l2==8: print('new term:',K[i,j,l2]*K[l2,k,l1],'\n\nrel0 =',rel)
                rel=rel.subs(J_test_subs)
                if rel!=0:
                    print(rel)

In [ ]:
for A in JR_K1:
    if JR_dict[A]!=JR_check[A]:
        print(A)
        print(JR_dict[A],'\n\n',JR_check[A],'\n\n\n')

In [ ]:
print(missing[0])
JR_check[missing[0]]

### Sanity Checks

In [ ]:
# Sanity check: is F1_N normal up to degree 1?
K1_N_p1=K1_N.wght_proj(1)
K1_N_Np=T.cochain_complex.subspace_proj(K1_N_p1,'im')[0]
simplify_cochain(K1_N_Np)
K1_N_Np # Should be zero

In [ ]:
# Sanity check: is K1 the str_func of F1?

for i in range(len(T.basis)):
    for j in range(i,len(T.basis)):
        ad_ij1=SF_ad(T.elt(list(F1.col(i))),T.elt(list(F1.col(j))),K0)
        w=zeros(len(T.basis),1)
        for k in range(len(T.basis)):
            key=(T.basis_strs[i],T.basis_strs[j],T.basis_strs[k])
            if key in K1.coeff_dict: 
                w+=F1.col(k)*K1.coeff_dict[key]
        ad_ij2=sum([w[l]*T.basis[l] for l in range(len(T.basis))])
        result=T.elt([simplify(ad_ij1.vec_rep[l]-ad_ij2.vec_rep[l]) for l in range(len(T.basis))])
        if result!=T.elt([0]*len(T.basis)):
            print('\n\n\nFailure at', (i,j),':')
            print('ad_ij1 =', ad_ij1,'\n\nad_ij2 =',ad_ij2)
            print('\nDifference:',result)

In [ ]:
# Sanity check: is K1_N really the str_func of F1_N?

for i in range(len(T.basis)):
    for j in range(i,len(T.basis)):
        ad_ij1=SF_ad(T.elt(list(F1_N.col(i))),T.elt(list(F1_N.col(j))),K0)
        w=zeros(len(T.basis),1)
        for k in range(len(T.basis)):
            key=(T.basis_strs[i],T.basis_strs[j],T.basis_strs[k])
            if key in K1_N.coeff_dict: 
                w+=F1_N.col(k)*K1_N.coeff_dict[key]
        ad_ij2=sum([w[l]*T.basis[l] for l in range(len(T.basis))])
        result=T.elt([simplify(ad_ij1.vec_rep[l]-ad_ij2.vec_rep[l]) for l in range(len(T.basis))])
        if result!=T.elt([0]*len(T.basis)):
            print('\n\n\nFailure at', (i,j),':')
            print('ad_ij1 =', ad_ij1,'\n\nad_ij2 =',ad_ij2)
            print('\nDifference:',result)

In [ ]:
# Sanity check: K1_N=K1+df1
Test0=K1_N-K1-f1.coboundary()

In [ ]:
simplify_cochain(K1)
simplify_cochain(K1_N)
df1=f1.coboundary()
simplify_cochain(df1)

In [ ]:
list(T0.coeff_dict.keys())[2]

In [ ]:
print(T0.coeff_dict[('e2','e6','E')])

In [ ]:
T0=K1_N-K1-f1
for i in range(10):
    print('\n',T0.coeff_dict[list(T0.coeff_dict.keys())[i]])

In [ ]:
simplify_cochain(K1_N_Np)
K1_N_Np

## Assistance for some Manual Computations

In [ ]:
for i in [3]:
    Ai=T.basis[i]
    for j in [4]:
        Aj=T.basis[j]
        for k in [5]:
            Ak=T.basis[k]
            R1=SF_ad(Ai,SF_ad(Aj,Ak,K0),K0)
            R2=SF_ad(SF_ad(Ai,Aj,K0),Ak,K0)+SF_ad(Aj,SF_ad(Ai,Ak,K0),K0)
            v=(R1-R2).vec_rep
            print(R1-R2)
            print(v[6])

In [ ]:
for i in range(5,2*m+4):
    print('\n',(3,4,i,i+1),':',JR_dict[(3,4,i,i+1)])

In [ ]:
for i in range(4,m+4):
    print('\n',(3,i,2*m+7-i,2*m+4),':',JR_dict[(3,i,2*m+7-i,2*m+4)])

In [ ]:
for i in range(5,m+4):
    print('\n',(4,i,2*m+7-i,2*m+4),':',JR_dict[(4,i,2*m+7-i,2*m+4)])

In [ ]:
K0.wght_proj(1)

In [ ]:
for A in JR_dict:
    print(A,'\n',JR_dict[A],'\n\n')
    

In [ ]:
for i in [4]:
    Ai=T.basis[i]
    for j in [5]:
        Aj=T.basis[j]
        for k in [8]:
            Ak=T.basis[k]
            print(Ai,Aj,Ak)
            R1=SF_ad(Ai,SF_ad(Aj,Ak,K0),K0)
            print('\nad(e1,ad(e2,e5)) :')
            for l in range(len(R1.vec_rep)):
                print(T.basis[l],'-->',R1.vec_rep[l])
            R2=SF_ad(SF_ad(Ai,Aj,K0),Ak,K0)+SF_ad(Aj,SF_ad(Ai,Ak,K0),K0)
            print('\nad(ad(e1,e2),e5) + ad(e2,ad(e2,e5)) =')
            for l in range(len(R2.vec_rep)):
                print(T.basis[l],'-->',R1.vec_rep[l])
            v=(R1-R2).vec_rep
            print('\nDifference:')
            for l in range(len((R1-R2).vec_rep)):
                print(T.basis[l],'-->',R1.vec_rep[l])

## Testing the projection

In [ ]:
def random_cochain(C,deg):
    result=C.cochain({})
    k=random.randint(1,9)
    for j in range(k):
        A=[T.basis_strs[random.randint(0,len(T.basis)-1)] for i in range(deg+1)]
        result+=C.cochain({tuple(A):random.randint(-15,15)})
    return result

In [ ]:
# Let's take a collection of random 1-cochains to get exact 2-cochains.
# Then, we'll make sure these project to themselves under the image projection

for i in range(10):
    curr=random_cochain(C,1)
    print(curr)
    d_curr=curr.coboundary()

In [ ]:
CH_1=CH_1.subs(prenorm_dict)
g,g_coords=C7.subspace_proj(CH_1,'im')
f=C7.preim_elt(g_coords)
CH_2 = CH_1-g #CH_2 is normal

In [ ]:
# Let's make a sanity check:
print(C7.coboundary(f)==g)
print(C7.subspace_proj(g,'im')[0]==g)
print(C7.coker_proj(CH_2)==CH_2)

## Mini tests

In [ ]:
# is_poly test:
t0=0
t1=1
t2=K[0]
t3=K[0]*K[1]
t4=K[1]**2
t5=K[1]+K[2]
t6=K[1]-K[2]
t7=K[1]*K[2]+K[3]**2
t8=sqrt(2)
t9=sqrt(2)*K[0]
t10=4*K[1]-(1/3)*K[2]
t11=4*K[1]-Rational(1,3)*K[2]
t12=4*K[1]-(1/2)*K[2]
t13=4*K[1]-Rational(1,2)*K[2]
t14=symbols('x')

r0=sqrt(2*K[0])
r1=K[0]**(1/2)
r2=K[0]**(1/3)
r3=sin(K[0])
r4=K[0]/sqrt(K[0])
r5=1+K[0]+sqrt(K[1])
r6=K[0]+sqrt(K[0])
r7=sqrt(K[0]**2+sqrt(K[1]+5*K[2]))
r8=K[1]/K[2]

for t in [t0,t1,t2,t3,t4,t5,t6,t7,t8,t9,t10,t11,t12,t13,t14]:
    if not is_poly(t):
        print('Failure:',t,'returns False')
for r in [r0,r1,r2,r3,r4,r5,r6,r7,r8]:
    if is_poly(r):
        print('Failure:',r,'returns True')

In [ ]:
# str_func test

TF1=eye(len(T.basis))
for i in range(2,len(T.basis)):
    TF1[i-2,i]=1
TK1=str_func(T,TF1,K0)

for i in range(len(T.basis)):
    for j in range(i,len(T.basis)):
        ad_ij1=SF_ad(T.elt(list(TF1.col(i))),T.elt(list(TF1.col(j))),K0)
        w=zeros(len(T.basis),1)
        for k in range(len(T.basis)):
            key=(T.basis_strs[i],T.basis_strs[j],T.basis_strs[k])
            if key in TK1.coeff_dict: 
                w+=Matrix([A*TK1.coeff_dict[key] for A in TF1.col(k)])       
        ad_ij2=sum([w[l]*T.basis[l] for l in range(len(T.basis))])
        if ad_ij1!=ad_ij2:
            print('\n\n\nFailure at', (i,j),':')
            print('ad_ij1 =', ad_ij1,'\n\nad_ij2 =',ad_ij2)
            print('\nDifference:',ad_ij1-ad_ij2)

In [ ]:
# SF_ad test

T_SF=C.cochain()
for i in range(len(T.basis)):
    for j in range(i,len(T.basis)):
        v=T.basis[i].ad(T.basis[j])
        for k in range(len(T.basis)):
            T_SF=T_SF+C.cochain({(T.basis_strs[i],T.basis_strs[j],T.basis_strs[k]):v.vec_rep[k]})

# Test: Does T_SF give the same ad as ad does?

A=4*T.basis[0]-2*T.basis[3]
B=T.basis[1]+3*T.basis[7]+T.basis[3]
A.ad(B)==SF_ad(A,B,T_SF)

In [ ]:
# SF_ad test

L=IndexedBase('L')

TK=C.cochain()
for i in range(len(T.basis)):
    for j in range(i,len(T.basis)):
        v=T.basis[i].ad(T.basis[j])
        for k in range(len(T.basis)):
            TK=TK+C.cochain({(T.basis_strs[i],T.basis_strs[j],T.basis_strs[k]):v.vec_rep[k]})
for i in range(6):
    TK=TK+C.cochain({(T.basis_strs[3],T.basis_strs[9],T.basis_strs[4+i]):L[3,9,4+i]})
    
# Another setup for manual testing
h,e,y=symbols('h,e,y')
Y=T.basis[0]
H=T.basis[1]
E=T.basis[2]
X=T.basis[3]
e5=L[1,1,1]**2*T.basis[8]
e6=T.basis[9]
SF_ad(E,e*e5,TK)

#TK

In [ ]:
## class ind_der:
#     def __init__(self,ind_obj,der_list=[]):
#         self.ind_obj=ind_obj
#         self.der_mset=Multiset(der_list)
        
#     def __eq__(self,other):
#         return (self.ind_obj == other.ind_obj and self.der_set == other.der_set)
        
#     def __add__(self,other):
#         if type(other)==ind_der:
#             return ind_mul(self)+other
#         if type(other)==ind_mul:
#             return other+self
#         if type(other)==ind_add:
#             return other+self
            
#     def __mul__(self,other):
#         if type(other)==int or type(other)==float:
#             return ind_mul({self:1},other)
#         if type(other)==ind_der:
#             return ind_mul(self)*other
#         if type(other)==ind_mul:
#             return other*self
#         if type(other)==ind_add:
#             return other*self
    
#     def __rmul__(self,other):
#         return self*other
            
#     def __radd__(self,other):
#         return self+other
    
#     def __neg__(self):
#         return -ind_mul(self)
        
#     def __sub__(self,other):
#         return other+(-self)
    
#     def __str__(self):
#         der_str=str(self.der_mset)
#         ind_ob_str=str(self.ind_obj)
#         return ind_ob_str[:len(ind_ob_str)-1]+'; '+der_str[1:len(der_str)-1]+ind_ob_str[len(ind_ob_str)-1:]
    
#     def __repr__(self):
#         der_str=str(self.der_mset)
#         ind_ob_str=str(self.ind_obj)
#         return ind_ob_str[:len(ind_ob_str)-1]+'; '+der_str[1:len(der_str)-1]+ind_ob_str[len(ind_ob_str)-1:]
    